# Этап 3 (v3): Адаптивный семантический анализ курсовой на FRIDA

Эта версия не привязана к конкретному количеству пунктов (3/4/5+).

Идея v3:
- строить метрики по реальной структуре документа из `meta.json`;
- использовать адаптивные (относительные) пороги;
- выдавать не только общий риск, но и локализованные проблемы.


## 1) Импорты и конфигурация

В этом блоке задаем пути, технические параметры и фиксируемые пороги.

Почему это важно:
- пороги не должны "плавать" под каждый документ;
- так метрики более переносимы между работами.


In [1]:
#  1:    .
import json
from pathlib import Path

import numpy as np

BASE = Path('out')
META_PATH = BASE / 'meta.json'
EMB_PATH = BASE / 'emb_FRIDA.npy'
REPORT_PATH = BASE / 'document_metrics_report_v3.json'

# : .   markdown- .
TOP_N_TRANSITIONS = 7
TOP_N_OUTLIERS = 7
TOP_N_ISSUES = 12

# : .   markdown- .
REDUNDANCY_THRESHOLD = 0.90
COVERAGE_THRESHOLD = 0.72
NOVELTY_LOW_THRESHOLD = 0.58
NOVELTY_HIGH_THRESHOLD = 0.88

# : .   markdown- .
ADJ_WEAK_Z = -1.0
OUTLIER_Z = -1.0

print('meta  :', META_PATH)
print('emb   :', EMB_PATH)
print('report:', REPORT_PATH)


meta  : out\meta.json
emb   : out\emb_FRIDA.npy
report: out\document_metrics_report_v3.json


## 2) Загрузка данных и базовая структура

Здесь мы:
- считываем `meta.json` и `emb_FRIDA.npy`;
- строим cosine-матрицу `S`;
- извлекаем роли (`intro/main/conclusion`) и группировку по `parent`.


In [2]:
#  2:      .
with open(META_PATH, 'r', encoding='utf-8') as f:
    meta = json.load(f)

emb = np.load(EMB_PATH).astype(np.float32)

n_segments = len(meta)
if emb.shape[0] != n_segments:
    raise ValueError(f'Mismatch: emb={emb.shape[0]} meta={n_segments}')


def cosine_matrix(vectors: np.ndarray) -> np.ndarray:
    vectors = vectors / (np.linalg.norm(vectors, axis=1, keepdims=True) + 1e-9)
    return vectors @ vectors.T


S = cosine_matrix(emb)
roles = np.array([m.get('role', '') for m in meta])
paths = [str(m.get('path', '')) for m in meta]
parents = np.array([str(m.get('parent', '')) for m in meta])

main_idx = np.where(roles == 'main')[0]
intro_idx = np.where(roles == 'intro')[0]
conc_idx = np.where(roles == 'conclusion')[0]

intro_i = int(intro_idx[0]) if intro_idx.size else None
conc_i = int(conc_idx[0]) if conc_idx.size else None

parent_to_indices = {}
for i, p in enumerate(parents):
    parent_to_indices.setdefault(p, []).append(i)

print('segments:', n_segments)
print('S shape :', S.shape)
print('main/int/con:', len(main_idx), intro_i, conc_i)
print('parent groups:', len(parent_to_indices))


segments: 11
S shape : (11, 11)
main/int/con: 9 0 10
parent groups: 5


## 3) Диагностические метрики (адаптивный слой)

Это главный вычислительный блок. Он не делает выводов, а только считает сигналы:

- локальная связность и слабые переходы;
- semantic drift и smoothness;
- centrality/outlierness без self-similarity;
- coverage intro -> main/conclusion;
- novelty/redundancy по последовательности;
- межсекционные переходы (адаптивно по реальному числу разделов).


In [3]:
#  3:   .

def upper_triangle_values(M: np.ndarray) -> np.ndarray:
    if M.shape[0] < 2:
        return np.array([], dtype=np.float32)
    iu = np.triu_indices_from(M, k=1)
    return M[iu].astype(np.float32)

# : .   markdown- .
adj = np.array([float(S[i, i + 1]) for i in range(n_segments - 1)], dtype=np.float32)
adj_mean = float(adj.mean()) if adj.size else float('nan')
adj_std = float(adj.std(ddof=0)) if adj.size else float('nan')
adj_z = (adj - adj_mean) / adj_std if np.isfinite(adj_std) and adj_std > 0 else np.zeros_like(adj)

weak_transition_indices = [int(i) for i, z in enumerate(adj_z) if float(z) <= ADJ_WEAK_Z]
weakest_order = np.argsort(adj)[: min(TOP_N_TRANSITIONS, adj.size)] if adj.size else np.array([], dtype=np.int64)

weakest_transitions = []
for i in weakest_order:
    i = int(i)
    weakest_transitions.append(
        {
            'from_index': i,
            'to_index': i + 1,
            'from_path': paths[i],
            'to_path': paths[i + 1],
            'adj_similarity': float(adj[i]),
            'adj_z': float(adj_z[i]),
        }
    )

# : .   markdown- .
main_main_mean = float('nan')
intro_to_main_mean = float('nan')
conc_to_main_mean = float('nan')
intro_delta = float('nan')
conc_delta = float('nan')

if main_idx.size:
    mm_vals = upper_triangle_values(S[np.ix_(main_idx, main_idx)])
    if mm_vals.size:
        main_main_mean = float(mm_vals.mean())

if intro_i is not None and main_idx.size:
    intro_to_main_mean = float(S[intro_i, main_idx].mean())
if conc_i is not None and main_idx.size:
    conc_to_main_mean = float(S[conc_i, main_idx].mean())

if np.isfinite(intro_to_main_mean) and np.isfinite(main_main_mean):
    intro_delta = float(intro_to_main_mean - main_main_mean)
if np.isfinite(conc_to_main_mean) and np.isfinite(main_main_mean):
    conc_delta = float(conc_to_main_mean - main_main_mean)

# : .   markdown- .
redund_90 = float('nan')
if main_idx.size >= 2:
    mm_vals = upper_triangle_values(S[np.ix_(main_idx, main_idx)])
    if mm_vals.size:
        redund_90 = float(np.mean(mm_vals > REDUNDANCY_THRESHOLD))

# ---- Drift + smoothness ----
step_vecs = emb[1:] - emb[:-1]
step_norms = np.linalg.norm(step_vecs, axis=1) if step_vecs.size else np.array([], dtype=np.float32)

turn_sharpness = []
for i in range(len(step_vecs) - 1):
    a = step_vecs[i]
    b = step_vecs[i + 1]
    denom = (np.linalg.norm(a) * np.linalg.norm(b)) + 1e-9
    turn_sharpness.append(1.0 - float(np.dot(a, b) / denom))
turn_sharpness = np.array(turn_sharpness, dtype=np.float32)

step_mean = float(step_norms.mean()) if step_norms.size else 0.0
turn_mean = float(turn_sharpness.mean()) if turn_sharpness.size else 0.0
semantic_drift_score = float(0.6 * step_mean + 0.4 * turn_mean)

second_diff_norms = np.linalg.norm(emb[2:] - 2 * emb[1:-1] + emb[:-2], axis=1) if n_segments >= 3 else np.array([], dtype=np.float32)
second_diff_mean = float(second_diff_norms.mean()) if second_diff_norms.size else 0.0

skip2 = np.array([float(S[i, i + 2]) for i in range(n_segments - 2)], dtype=np.float32) if n_segments >= 3 else np.array([], dtype=np.float32)
skip2_mean = float(skip2.mean()) if skip2.size else float('nan')

smoothness_score = float((1.0 / (1.0 + second_diff_mean)) * 0.5 + (skip2_mean if np.isfinite(skip2_mean) else 0.0) * 0.5)

# : .   markdown- .
S_wo_diag = S.copy().astype(np.float32)
np.fill_diagonal(S_wo_diag, np.nan)
centrality_all = np.nanmean(S_wo_diag, axis=1).astype(np.float32)

c_mean = float(np.nanmean(centrality_all))
c_std = float(np.nanstd(centrality_all))
centrality_z = (centrality_all - c_mean) / c_std if c_std > 0 else np.zeros_like(centrality_all)
outlierness = (-centrality_z).astype(np.float32)

outlier_order = np.argsort(outlierness)[::-1][: min(TOP_N_OUTLIERS, n_segments)]
segment_outliers = []
for idx in outlier_order:
    idx = int(idx)
    segment_outliers.append(
        {
            'index': idx,
            'path': paths[idx],
            'role': str(roles[idx]),
            'centrality_all': float(centrality_all[idx]),
            'centrality_z': float(centrality_z[idx]),
            'outlierness': float(outlierness[idx]),
        }
    )

# ---- Coverage ----
intro_to_main_best = []
intro_to_conc_best = []
for i in intro_idx:
    i = int(i)
    if main_idx.size:
        intro_to_main_best.append(float(np.max(S[i, main_idx])))
    if conc_idx.size:
        intro_to_conc_best.append(float(np.max(S[i, conc_idx])))

intro_to_main_best = np.array(intro_to_main_best, dtype=np.float32)
intro_to_conc_best = np.array(intro_to_conc_best, dtype=np.float32)

coverage_main = float(np.mean(intro_to_main_best >= COVERAGE_THRESHOLD)) if intro_to_main_best.size else float('nan')
coverage_conclusion = float(np.mean(intro_to_conc_best >= COVERAGE_THRESHOLD)) if intro_to_conc_best.size else float('nan')
coverage_parts = [x for x in [coverage_main, coverage_conclusion] if np.isfinite(x)]
coverage_score = float(np.mean(coverage_parts)) if coverage_parts else float('nan')

# ---- Novelty/redundancy profile ----
novelty_profile = []
for i in range(1, n_segments):
    sim_prev = float(np.max(S[i, :i]))
    if sim_prev >= NOVELTY_HIGH_THRESHOLD:
        label = 'redundant'
    elif sim_prev <= NOVELTY_LOW_THRESHOLD:
        label = 'jump'
    else:
        label = 'balanced'
    novelty_profile.append(
        {
            'index': i,
            'path': paths[i],
            'max_similarity_to_previous': sim_prev,
            'label': label,
        }
    )

redundant_share = float(np.mean([x['label'] == 'redundant' for x in novelty_profile])) if novelty_profile else float('nan')
jump_share = float(np.mean([x['label'] == 'jump' for x in novelty_profile])) if novelty_profile else float('nan')

# : .   markdown- .
section_boundaries = []
for i in range(n_segments - 1):
    if parents[i] != parents[i + 1]:
        section_boundaries.append(i)

boundary_transitions = []
for i in section_boundaries:
    boundary_transitions.append(
        {
            'from_index': int(i),
            'to_index': int(i + 1),
            'from_parent': str(parents[i]),
            'to_parent': str(parents[i + 1]),
            'similarity': float(S[i, i + 1]),
            'z': float(adj_z[i]) if i < len(adj_z) else None,
            'from_path': paths[i],
            'to_path': paths[i + 1],
        }
    )

print('adj_mean:', round(adj_mean, 4))
print('semantic_drift_score:', round(semantic_drift_score, 4))
print('smoothness_score:', round(smoothness_score, 4))
print('coverage_score:', round(coverage_score, 4) if np.isfinite(coverage_score) else 'nan')
print('redundant_share/jump_share:',
      round(redundant_share, 4) if np.isfinite(redundant_share) else 'nan', '/',
      round(jump_share, 4) if np.isfinite(jump_share) else 'nan')
print('section boundaries:', len(boundary_transitions))


adj_mean: 0.5731
semantic_drift_score: 1.1328
smoothness_score: 0.4859
coverage_score: 0.5
redundant_share/jump_share: 0.1 / 0.1
section boundaries: 4


## 4) Правила интерпретации (слой решений)

Здесь мы превращаем метрики в понятные диагностические события:
- общие флаги (слабая связность, дрейф, coverage, избыточность);
- локализованные issues по сегментам/переходам;
- severity (high/medium/low).


In [4]:
#  4:     .
problem_flags = []
issues = []

# : .   markdown- .
if np.isfinite(adj_mean) and adj_mean < 0.65:
    problem_flags.append({'type': 'low_local_coherence', 'severity': 'high', 'metric': 'adj_mean', 'value': float(adj_mean), 'threshold': 0.65})

if np.isfinite(coverage_score) and coverage_score < 0.50:
    problem_flags.append({'type': 'low_goal_coverage', 'severity': 'high', 'metric': 'coverage_score', 'value': float(coverage_score), 'threshold': 0.50})

if np.isfinite(conc_delta) and conc_delta < -0.05:
    problem_flags.append({'type': 'conclusion_main_misalignment', 'severity': 'medium', 'metric': 'conclusion_delta', 'value': float(conc_delta), 'threshold': -0.05})

if np.isfinite(redund_90) and redund_90 > 0.25:
    problem_flags.append({'type': 'high_redundancy', 'severity': 'medium', 'metric': 'redund_90', 'value': float(redund_90), 'threshold': 0.25})

if np.isfinite(jump_share) and jump_share > 0.30:
    problem_flags.append({'type': 'high_semantic_jumps', 'severity': 'medium', 'metric': 'jump_share', 'value': float(jump_share), 'threshold': 0.30})

# : .   markdown- .
for tr in weakest_transitions:
    sev = 'high' if tr['adj_z'] <= -1.5 else ('medium' if tr['adj_z'] <= -1.0 else 'low')
    issues.append(
        {
            'type': 'weak_transition',
            'severity': sev,
            'where': f"{tr['from_index']}->{tr['to_index']}",
            'from_path': tr['from_path'],
            'to_path': tr['to_path'],
            'metric': 'adj_similarity',
            'value': float(tr['adj_similarity']),
            'aux_z': float(tr['adj_z']),
                'recommendation': '    /   .',
        }
    )

# : .   markdown- .
for s in segment_outliers:
    if s['centrality_z'] <= OUTLIER_Z:
        sev = 'high' if s['centrality_z'] <= -1.5 else 'medium'
        issues.append(
            {
                'type': 'outlier_segment',
                'severity': sev,
                'where': f"index={s['index']}",
                'from_path': s['path'],
                'to_path': None,
                'metric': 'centrality_z',
                'value': float(s['centrality_z']),
                'aux_z': float(s['outlierness']),
                'recommendation': '    /   .',
            }
        )

# : .   markdown- .
for b in boundary_transitions:
    z = b['z'] if b['z'] is not None else 0.0
    if z <= -1.0:
        sev = 'high' if z <= -1.5 else 'medium'
        issues.append(
            {
                'type': 'weak_section_boundary',
                'severity': sev,
                'where': f"{b['from_index']}->{b['to_index']}",
                'from_path': b['from_path'],
                'to_path': b['to_path'],
                'metric': 'boundary_similarity',
                'value': float(b['similarity']),
                'aux_z': float(z),
                'recommendation': '    /   .',
            }
        )

# : .   markdown- .
rank = {'high': 0, 'medium': 1, 'low': 2}
issues = sorted(issues, key=lambda x: (rank.get(x['severity'], 9), x.get('value', 1.0)))
issues = issues[:TOP_N_ISSUES]

print('global flags:', len(problem_flags))
print('local issues:', len(issues))


global flags: 2
local issues: 11


## 5) Финальный отчет и risk score

Здесь мы собираем итоговый JSON.

Важно: `risk_score_v3` вспомогательный. Главное для интерпретации — список локальных `issues` и их причины.


In [5]:
#  5:     risk score.
risk_score = 0.0
if np.isfinite(adj_mean):
    risk_score += max(0.0, (0.70 - adj_mean)) * 100.0
if np.isfinite(semantic_drift_score):
    risk_score += min(20.0, semantic_drift_score * 15.0)
if np.isfinite(smoothness_score):
    risk_score += max(0.0, (0.60 - smoothness_score)) * 35.0
if np.isfinite(coverage_score):
    risk_score += max(0.0, (0.70 - coverage_score)) * 45.0
if np.isfinite(redund_90):
    risk_score += max(0.0, redund_90 - 0.10) * 60.0
if np.isfinite(jump_share):
    risk_score += jump_share * 30.0

risk_score = float(min(100.0, max(0.0, risk_score)))

report = {
    'version': 'v3',
    'model': 'FRIDA',
    'n_segments': int(n_segments),
    'diagnostic_metrics': {
        'local_coherence': {
            'adj_mean': float(adj_mean),
            'adj_std': float(adj_std),
            'weak_transition_count': int(len(weak_transition_indices)),
            'weak_transition_share': float(len(weak_transition_indices) / adj.size) if adj.size else None,
            'weakest_transitions_top_n': weakest_transitions,
        },
        'structure_consistency': {
            'intro_to_main_mean': float(intro_to_main_mean) if np.isfinite(intro_to_main_mean) else None,
            'conclusion_to_main_mean': float(conc_to_main_mean) if np.isfinite(conc_to_main_mean) else None,
            'main_main_mean': float(main_main_mean) if np.isfinite(main_main_mean) else None,
            'intro_delta': float(intro_delta) if np.isfinite(intro_delta) else None,
            'conclusion_delta': float(conc_delta) if np.isfinite(conc_delta) else None,
        },
        'drift_and_smoothness': {
            'semantic_drift_score': float(semantic_drift_score),
            'step_mean_norm': float(step_mean),
            'turn_mean_sharpness': float(turn_mean),
            'second_diff_mean': float(second_diff_mean),
            'skip2_mean': float(skip2_mean) if np.isfinite(skip2_mean) else None,
            'smoothness_score': float(smoothness_score),
        },
        'centrality': {
            'mean': float(np.nanmean(centrality_all)),
            'std': float(np.nanstd(centrality_all)),
            'top_outliers': segment_outliers,
        },
        'coverage': {
            'threshold': float(COVERAGE_THRESHOLD),
            'coverage_main': float(coverage_main) if np.isfinite(coverage_main) else None,
            'coverage_conclusion': float(coverage_conclusion) if np.isfinite(coverage_conclusion) else None,
            'coverage_score': float(coverage_score) if np.isfinite(coverage_score) else None,
        },
        'novelty_redundancy_profile': {
            'low_threshold': float(NOVELTY_LOW_THRESHOLD),
            'high_threshold': float(NOVELTY_HIGH_THRESHOLD),
            'redundant_share': float(redundant_share) if np.isfinite(redundant_share) else None,
            'jump_share': float(jump_share) if np.isfinite(jump_share) else None,
            'items': novelty_profile,
        },
        'section_boundaries': boundary_transitions,
        'redundancy': {
            'threshold': float(REDUNDANCY_THRESHOLD),
            'redund_90': float(redund_90) if np.isfinite(redund_90) else None,
        }
    },
    'interpretation': {
        'global_flags': problem_flags,
        'localized_issues': issues,
    },
    'summary': {
        'risk_score_v3': float(risk_score),
        'note': 'Analyze localized_issues first, then global_flags. risk_score_v3 is supportive.',
    }
}

REPORT_PATH.parent.mkdir(parents=True, exist_ok=True)
REPORT_PATH.write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding='utf-8')

print('saved:', REPORT_PATH)
print('risk_score_v3:', round(risk_score, 2))


saved: out\document_metrics_report_v3.json
risk_score_v3: 45.67


## 6) Быстрая сводка для ручной проверки

Эта ячейка печатает короткий итог, чтобы можно было быстро оценить качество документа.


In [6]:
#  6:   .
print(':', report['model'])
print(':', report['version'])
print(':', report['n_segments'])
print('RISK SCORE V3:', round(report['summary']['risk_score_v3'], 2))

print('')
print(' :')
print('- adj_mean:', round(report['diagnostic_metrics']['local_coherence']['adj_mean'], 4))
print('- semantic_drift_score:', round(report['diagnostic_metrics']['drift_and_smoothness']['semantic_drift_score'], 4))
print('- smoothness_score:', round(report['diagnostic_metrics']['drift_and_smoothness']['smoothness_score'], 4))
print('- coverage_score:', report['diagnostic_metrics']['coverage']['coverage_score'])
print('- redundant_share:', report['diagnostic_metrics']['novelty_redundancy_profile']['redundant_share'])
print('- jump_share:', report['diagnostic_metrics']['novelty_redundancy_profile']['jump_share'])

print('')
print(' :')
if report['interpretation']['global_flags']:
    for f in report['interpretation']['global_flags']:
        print('-', f['type'], '|', f['severity'], '|', f['metric'], '=', round(float(f['value']), 4))
else:
    print('- ')

print('')
print('  :')
if report['interpretation']['localized_issues']:
    for it in report['interpretation']['localized_issues'][:TOP_N_ISSUES]:
        print('-', it['type'], '|', it['severity'], '|', it['where'])
        print('  metric:', it['metric'], '=', round(float(it['value']), 4))
        print('  rec   :', it['recommendation'])
else:
    print('- ')


: FRIDA
: v3
: 11
RISK SCORE V3: 45.67

 :
- adj_mean: 0.5731
- semantic_drift_score: 1.1328
- smoothness_score: 0.4859
- coverage_score: 0.5
- redundant_share: 0.1
- jump_share: 0.1

 :
- low_local_coherence | high | adj_mean = 0.5731
- conclusion_main_misalignment | medium | conclusion_delta = -0.0504

  :
- outlier_segment | medium | index=9
  metric: centrality_z = -1.3003
  rec   :     /   .
- outlier_segment | medium | index=6
  metric: centrality_z = -1.2667
  rec   :     /   .
- outlier_segment | medium | index=10
  metric: centrality_z = -1.0008
  rec   :     /   .
- weak_transition | medium | 9->10
  metric: adj_similarity = 0.3244
  rec   :     /   .
- weak_section_boundary | medium | 9->10
  metric: boundary_similarity = 0.3244
  rec   :     /   .
- weak_transition | medium | 5->6
  metric: adj_similarity = 0.3632
  rec   :     /   .
- weak_transition | low | 6->7
  metric: adj_similarity = 0.4606
  rec   :     /   .
- weak_transition | low | 1->2
  metric: adj_similarity =